# **Middleware**
----
**Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:**

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

## **Summarization Middleware**
---
**Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:**

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

#### **Summarization based on number of messages**

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

llm = ChatGroq(model="qwen/qwen3-32b")

agent = create_agent(
    model = llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages", 10),
            keep=("messages", 5)
        )
    ]
) 

In [4]:
config={"configurable":{"thread_id":"test-1"}}

In [7]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='7bc776fe-6633-423d-949b-21d9dfb9810c'), AIMessage(content='<think>\nOkay, so the user asked, "What is 2+2?" Hmm, that\'s a basic arithmetic question. Let me make sure I understand it correctly. They want to know the sum of two and two. Alright, in elementary math, adding two numbers together. Let me start by recalling the basic addition facts.\n\n2 plus 2... Well, if I count two objects and add another two, how many do I have in total? Let\'s visualize it. If I have two apples and someone gives me two more apples, I now have four apples. So 2 + 2 equals 4. That seems straightforward.\n\nWait, is there any chance they\'re looking for something more complex? Like in different number bases? For example, in base 3, 2 + 2 might be different. But the question doesn\'t specify a base, so I should assume base 10, which is the standard decimal system. In base 10, 2 + 2 is definitely 4. \

#### **Summarization based on token limits**

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

llm = ChatGroq(model="qwen/qwen3-32b")

agent = create_agent(
    model = llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 590),
            keep=("tokens", 230)
        )
    ]
) 

In [9]:
config={"configurable":{"thread_id":"test-1"}}

In [10]:
# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [11]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~1181 tokens, 2 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='f5e164f0-929f-4b69-88a8-2b1932a48e97'), AIMessage(content="<think>\nOkay, the user wants to find hotels in Paris. Let me think about how to approach this. First, I need to consider what they might need. Are they looking for budget options, luxury hotels, or something in between? Maybe they have specific preferences like location, amenities, or accessibility. \n\nI should start by mentioning some popular areas in Paris where hotels are concentrated, like the 1st to 12th arrondissements. Areas near major attractions such as the Eiffel Tower, Louvre, or Paris Opera are usually good spots. But I should also suggest using online booking platforms because they can filter by price, ratings, and amenities. Websites like Booking.com, Expedia, or Airbnb come to mind.\n\nWait, the user might not know about these platforms. I should list a few options and maybe explain how 

#### **Human in the loop MiddleWare**
**Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:**

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [13]:
agent=create_agent(
    model=llm,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [14]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [15]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='8fc04301-b77b-4632-bad0-581019e84444'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three pieces of information: recipient is john@test.com, subject is Hello, and body is How are you?. So I need to call the send_email_tool with these arguments. I should make sure there's no missing data. Everything seems there. So the correct function call is send_email_tool with those parameters.\n", 'tool_calls': [{'id': 'gj5nna7dr', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 

In [ ]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been successfully sent to john@test.com with the subject "Hello" and the message "How are you?". Let me know if there's anything else you need!


##### **Rejecting**

In [19]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [20]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Rejecting...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Rejecting...
✅ Result: The request to send an email to **john@test.com** with subject "Hello" and body "How are you?" was denied. Would you like to retry sending this email?


In [21]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='a13c8caa-96e2-46ba-b254-1160e709469e'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools.\n\nLooking at the tools provided, there's a send_email_tool. Its parameters are recipient, subject, body, all required. The function read_email_tool is for reading by ID, which isn't needed here.\n\nSo, I need to call send_email_tool with the given parameters. The recipient is john@test.com, subject is Hello, body is How are you?. I should structure the JSON arguments correctly. Make sure the keys are recipient, subject, body and the values are strings. No need for any other parameters. That should do it.\n", 'tool_calls': [{'id': '3grhrrjb8', 'function': {'arguments': '{"body":"How are you

##### **Editing**

In [22]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [23]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='0078878e-6080-4fa5-be2b-7081ec1d7196'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to wrong@email.com with the subject 'Test' and body 'Hello'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. The parameters are all there: recipient is the email address provided, subject is 'Test', and body is 'Hello'. I need to make sure all required fields are included. Yep, all three are required and present. I'll call the send_email_tool with those parameters.\n", 'tool_calls': [{'id': '7d8gegqve', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 249, 'total_tokens'

In [24]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: The email has been successfully sent to **correct@email.com** with the subject **"Corrected Subject"**. The body was edited to:  
*"This was edited by human before sending"*  

Let me know if you need anything else!


In [25]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='0078878e-6080-4fa5-be2b-7081ec1d7196'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to wrong@email.com with the subject 'Test' and body 'Hello'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. The parameters are all there: recipient is the email address provided, subject is 'Test', and body is 'Hello'. I need to make sure all required fields are included. Yep, all three are required and present. I'll call the send_email_tool with those parameters.\n", 'tool_calls': [{'id': '7d8gegqve', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 249, 'total_tokens'

## **TO-DO-LIST Middleware**

**Equip agents with task planning and tracking capabilities for complex multi-step tasks. To-do lists are useful for the following:**

- Complex multi-step tasks requiring coordination across multiple tools.
- Long-running operations where progress visibility is important.

In [30]:
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import HumanMessage
from langchain.tools import tool
import os
import subprocess
import tempfile

@tool("read_file")
def read_file(file_path: str):
    """function to read a file."""
    with open(file_path, "r") as f:
        return f.read()

@tool("write_file")
def write_file(file_path: str, content: str):
    """function to write content to a file."""
    with open(file_path, "w") as f:
        f.write(content)
    return f"File '{file_path}' written successfully."

@tool("run_tests")
def run_tests(test_command: str):
    """function to run tests using a shell command."""
    result = subprocess.run(test_command, shell=True, capture_output=True, text=True)
    return f"Test Output:\n{result.stdout}\nTest Errors:\n{result.stderr}"    

@tool
def make_directory(path: str):
    """Create directory if it doesn't exist."""
    os.makedirs(path, exist_ok=True)
    return f"Created {path}"

@tool
def run_command(command: str):
    """Run a shell command."""
    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True
    )
    return result.stdout + result.stderr

agent = create_agent(
    model=llm,
    tools=[read_file, write_file, run_tests, make_directory, run_command],
    middleware=[TodoListMiddleware()],
)

In [35]:
response = agent.invoke(
    {"messages": [HumanMessage(content="""
    create a file called index.html and make the beautiful minimalist ecommerce style home page and then start it
""")]},
)

response

{'messages': [HumanMessage(content='\n    create a file called index.html and make the beautiful minimalist ecommerce style home page and then start it\n', additional_kwargs={}, response_metadata={}, id='3c1b0a98-5509-4a74-aa0b-3a321559d26f'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to create an index.html file with a beautiful minimalist e-commerce style home page and then start it. Hmm, first, I need to make sure I understand the requirements. They mentioned "minimalist e-commerce," so I should focus on clean design, maybe a grid layout for products, a header with navigation, and some call-to-action buttons.\n\nFirst step, I need to create the index.html file. But wait, the user also said "start it." Starting a web page usually means running a local server to view it. Since I\'m working in an environment where I can execute commands, maybe I can set up a simple HTTP server. But how?\n\nI\'ll need to write the HTML content. 